# שיעור 10 - סוכני בינה מלאכותית בייצור

בשיעור זה תלמדו **דפוסי ייצור** עבור סוכני בינה מלאכותית באמצעות Microsoft Agent Framework (Python). נסקור:

- **תצפית** — הוספת מדידת זמן ורישום לאינטראקציות של הסוכן
- **הערכה** — שימוש בסוכן מעריך כדי לדרג את איכות התשובות
- **ניהול עלויות** — אסטרטגיות לאופטימיזציה של טוקנים ובחירת מודל

התסריט הוא של **סוכן נסיעות** שעוזר למשתמשים לתכנן טיולים, כאשר ניטור והערכה משולבים מעל.


## התקנה


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## שיקולים בסביבת הייצור

העברה של סוכני בינה מלאכותית מפרוטוטיפים לסביבת הייצור דורשת תשומת לב קפדנית לשלושת עמודי התווך:

1. **תצפית** — יש צורך בנראות לגבי מה שהסוכן עושה, כמה זמן זה לוקח, ואילו כלים הוא מפעיל. ללא מעקב ורישום לוגים, ניפוי באגים של תקלות בסביבת הייצור כמעט בלתי אפשרי.

2. **הערכה** — בדיקות איכות אוטומטיות מבטיחות שתשובות הסוכן יישארו מדויקות, שלמות ומועילות לאורך זמן. סוכן מעריך יכול לתת ניקוד לתשובות על פי קריטריונים מוגדרים.

3. **ניהול עלויות** — שימוש בטוקנים משפיע ישירות על העלות. אסטרטגיות כמו אופטימיזציה של הפרומפט, בחירת מודל ושימוש במטמון עוזרות לשמור על ההוצאות תחת שליטה מבלי לפגוע באיכות.


## בניית סוכן הניתן למעקב

אנו מגדירים כלי נסיעה ועוטפים את קריאת הסוכן במדידת זמן כדי שנוכל לנטר את ההשהייה. בסביבת ייצור יש לשלב זאת עם OpenTelemetry או עם מערכת מעקב דומה.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## דפוסי הערכה

דפוס נפוץ בסביבת ייצור הוא להשתמש בסוכן שני כ-**מעריך**. המעריך מדרג את תגובת הסוכן הראשי בהתאם לקריטריונים שהוגדרו מראש, כגון שלמות, דיוק ושימושיות.

זה מאפשר:
- שערי איכות אוטומטיים לפני שהתגובות מגיעות למשתמשים
- זיהוי רגרסיה כאשר ההנחיות או המודלים משתנים
- ניטור מתמשך של ביצועי הסוכן לאורך זמן


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## אסטרטגיות לניהול עלויות

שליטה בעלויות קריטית עבור סוכני בינה מלאכותית בסביבות ייצור. להלן אסטרטגיות מרכזיות:

| אסטרטגיה | תיאור |
|---|---|
| **אופטימיזציה של הפרומפט** | שמרו על הוראות המערכת תמציתיות. הסירו קונטקסט מיותר כדי להפחית את כמות הטוקנים בקלט. |
| **בחירת מודל** | השתמשו במודלים קטנים וזולים יותר (למשל GPT-4o-mini) למשימות פשוטות כמו סיווג או חילוץ, והקצו מודלים גדולים יותר רק למשימות הדורשות חשיבה מורכבת. |
| **מטמון** | אחסנו תוצאות כלי ושאילתות תכופות במטמון כדי להימנע מקריאות API מיותרות. |
| **תקציבי טוקנים** | קבעו מגבלות של `max_tokens` כדי למנוע תגובות ארוכות באופן בלתי צפוי. |
| **עיבוד באצוות** | קבצו מספר שאילתות משתמש לקריאת API אחת כשאפשר. |

בפועל, גישה מדורגת עובדת היטב: הפנו בקשות פשוטות למודל מהיר וזול, והעבירו רק שאילתות מורכבות למודל בעל יכולות גבוהות יותר.


## סיכום

בשיעור זה למדת כיצד:

1. **להוסיף יכולת תצפית** לאינטראקציות של הסוכן באמצעות מדידת זמנים ורישום לוגים, ובכך להניח את היסוד לעקיבה ולניטור.
2. **להעריך אוטומטית תגובות של הסוכן** באמצעות סוכן מעריך שמדרג שלמות, דיוק ושימושיות.
3. **לנהל עלויות** באמצעות אופטימיזציה של ההנחיה, בחירת מודל, מטמון ותקציבי אסימונים.

תבניות ייצור אלה עוזרות להבטיח שהסוכנים של הבינה המלאכותית שלך יהיו אמינים, ניתנים למדידה וחסכוניים בקנה מידה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
הצהרת אחריות:
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator] (https://github.com/Azure/co-op-translator). אמנם אנו שואפים לדיוק, אך יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי־דיוקים. יש להסתמך על המסמך המקורי בשפת המקור כמקור הסמכות. למידע קריטי מומלץ לבצע תרגום מקצועי על ידי מתרגם אנושי. איננו אחראים לכל אי־הבנות או לפרשנויות מוטעות הנובעות משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
